# Tutorial 03: Binary-Domain Phase Space

This notebook explores the three-dimensional generator phase space

$$ (k_0, \epsilon, \langle u \rangle_{target}) $$

where `k0` controls the selected spatial frequency, `eps` controls the instability strength, and `target_mean` controls the balance between the two domain phases during evolution. Every image uses the generator's zero threshold followed only by the same bounded small-hole cleanup and saturation rule used by the production sweep.

> The generator is called with `region="custom"`; named presets would overwrite `k0` and `eps`.

In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from scipy import ndimage

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = repo_root.parent
src_dir = repo_root / "src"
if not src_dir.exists():
    raise RuntimeError(f"Cannot find repository src directory from {Path.cwd()}")
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from scattering_calculator.sample_generator.gray_scott_generator_binary import generate as generate_binary
from scattering_calculator.sample_generator.domain_analysis import classify_magnetic_domains
from scattering_calculator.sample_generator import fill_small_domain_holes
plt.rcParams["figure.constrained_layout.use"] = True

## 1. Scan settings

The same seed, image size, evolution time, and noise level are used throughout, so the three scanned parameters are the only systematic differences. Increase the number of values for a denser scan.

In [ ]:
shape = (128, 128)
n_steps = 80
noise_amp = 0.0  # fixed at zero so the coordinate map is exactly reproducible
seed = 7

k0_values = np.linspace(0.1, 1.1, 12)
eps_values = np.linspace(0.1, 1.2, 12)
target_mean_values = np.linspace(-1.0, 1.0, 12)

# Optional stricter filters; zero leaves any resolved bubble/stripe valid.
min_phase_fraction = 0.0  # optional stricter phase-occupancy filter
min_largest_component_fraction = 0.0  # optional stricter connectivity filter
min_boundary_density = 0.0  # optional stricter interface-density filter

# Bubble classification: compact, resolved, non-border minority components.
bubble_min_area = 9
bubble_max_eccentricity = 0.85
bubble_min_circularity = 0.45
saturation_fraction_threshold = 0.01
pattern_max_hole_area_px = 9

In [ ]:
def generate_one(k0, eps, target_mean):
    """Generate one raw continuous field and its zero-threshold binary output."""
    _, binary, continuous, metadata = generate_binary(
        batch=1, H=shape[0], W=shape[1], n_steps=n_steps,
        region="custom", use_gpu=False, seed=seed,
        k0=k0, eps=eps, target_mean=target_mean,
        noise_amp=noise_amp, quadratic_coefficient=0.0)
    binary = np.asarray(binary[0], dtype=float)
    negative_fraction = float(np.mean(binary < 0))
    if negative_fraction <= saturation_fraction_threshold:
        binary = np.ones_like(binary)
    elif negative_fraction >= 1.0 - saturation_fraction_threshold:
        binary = -np.ones_like(binary)
    binary = fill_small_domain_holes(binary, pattern_max_hole_area_px)
    return np.asarray(continuous[0]), binary, metadata

def down_fraction(binary):
    return float(np.mean(binary < 0))

def boundary_density(binary):
    horizontal = np.mean(binary[:, 1:] != binary[:, :-1])
    vertical = np.mean(binary[1:, :] != binary[:-1, :])
    return float(0.5 * (horizontal + vertical))

def largest_component_fraction(mask):
    """Fraction of one phase belonging to its largest connected component."""
    labels, count = ndimage.label(mask)
    if count == 0 or not np.any(mask):
        return 0.0
    areas = np.bincount(labels.ravel())[1:]
    return float(areas.max() / np.sum(mask))

def classify_domain_pattern(binary):
    """Evaluate optional occupancy, connectivity, and interface filters."""
    down = binary < 0
    fraction = down.mean()
    down_connectivity = largest_component_fraction(down)
    up_connectivity = largest_component_fraction(~down)
    edges = boundary_density(binary)
    valid = (
        min_phase_fraction <= fraction <= 1.0 - min_phase_fraction
        and down_connectivity >= min_largest_component_fraction
        and up_connectivity >= min_largest_component_fraction
        and edges >= min_boundary_density
    )
    return valid, down_connectivity, up_connectivity

## 2. Calculate the 3D phase space

Results are cached in a dictionary indexed by `(target_mean, eps, k0)`, so all later plots reuse exactly the same generated patterns.

In [ ]:
results = {}
for target_mean in target_mean_values:
    for eps in eps_values:
        for k0 in k0_values:
            field, binary, metadata = generate_one(k0, eps, target_mean)
            passes_optional_filters, down_connectivity, up_connectivity = classify_domain_pattern(binary)
            morphology = classify_magnetic_domains(
                binary, min_area=bubble_min_area,
                max_eccentricity=bubble_max_eccentricity,
                min_circularity=bubble_min_circularity)
            has_resolved_domains = (
                morphology["bubble_count"] >= 1 or morphology["stripe_count"] >= 1)
            valid = passes_optional_filters and has_resolved_domains
            results[(target_mean, eps, k0)] = {
                "field": field,
                "binary": binary,
                "down_fraction": down_fraction(binary),
                "boundary_density": boundary_density(binary),
                "down_connectivity": down_connectivity,
                "up_connectivity": up_connectivity,
                "valid_domain_pattern": valid,
                "morphology": morphology,
            }

print(f"Generated {len(results)} parameter combinations")

## 3. Morphology slices through the 3D space

Each figure is one constant-`target_mean` slice. Columns vary `k0`; rows vary `eps`. Positive `target_mean` makes the dark (`-1`) phase the minority, while negative values make it the majority.

In [ ]:
slice_mean_values = target_mean_values[::2]
for target_mean in slice_mean_values:
    fig, axes = plt.subplots(len(eps_values), len(k0_values), figsize=(10, 10))
    for row, eps in enumerate(eps_values):
        for col, k0 in enumerate(k0_values):
            result = results[(target_mean, eps, k0)]
            axes[row, col].imshow(result["binary"][:40,:40], cmap="gray", vmin=-1, vmax=1)
            #axes[row, col].set_title(
            #    f"k0={k0:.2f}, eps={eps:.2f}\ndown={result['down_fraction']:.0%}",
            #    fontsize=8)
            axes[row, col].set_axis_off()
    fig.suptitle(f"k0 × eps slice at target_mean = {target_mean:+.2f}", fontsize=14)
    plt.show()

## 4. 3D summary

The point position represents the three input parameters. Colour reports the resulting fraction of `-1` pixels, and point size reports boundary density as a simple measure of feature fineness/complexity.

In [ ]:
points = []
for (target_mean, eps, k0), result in results.items():
    points.append((k0, eps, target_mean, result["down_fraction"], result["boundary_density"]))
points = np.asarray(points)

fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection="3d")
sizes = 30 + 900 * points[:, 4]
scatter = ax.scatter(points[:, 0], points[:, 1], points[:, 2],
                     c=points[:, 3], s=sizes, cmap="viridis",
                     vmin=0, vmax=1, alpha=0.85)
ax.set_xlabel("k0")
ax.set_ylabel("eps")
ax.set_zlabel("target_mean")
fig.colorbar(scatter, ax=ax, label="down-domain fraction", shrink=0.7)
ax.set_title("Binary-domain 3D parameter space")

## 5. Occupancy curves

These curves show how the emergent binary area ratio changes with `target_mean` for every `(k0, eps)` pair. They make it easier to identify parameter regions where a small mean change causes a topology or occupancy transition.

In [ ]:
fig, axes = plt.subplots(1, len(eps_values), figsize=(15, 3.2), sharey=True)
for ax, eps in zip(axes, eps_values):
    for k0 in k0_values:
        fractions = [results[(mean, eps, k0)]["down_fraction"] for mean in target_mean_values]
        ax.plot(target_mean_values, fractions, marker="o", label=f"k0={k0:.2f}")
    ax.set_title(f"eps={eps:.2f}")
    ax.set_xlabel("target_mean")
    ax.grid(alpha=0.25)
axes[0].set_ylabel("down-domain fraction")
axes[-1].legend(fontsize=7, bbox_to_anchor=(1.03, 1), loc="upper left")

## 6. Valid-domain coordinate maps

A coordinate is marked **valid** when the cleaned pattern contains at least one resolved bubble or one resolved stripe-like component. A single bubble therefore counts. Phase occupancy, connectivity, and boundary-density filters remain available, but all three thresholds default to zero. Green cells are accepted coordinates; grey cells are saturated or contain only sub-resolution noise.

In [ ]:
ncols = 4
nrows = int(np.ceil(len(target_mean_values) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(12, 3 * nrows), squeeze=False)
validity_cube = np.zeros((len(target_mean_values), len(eps_values), len(k0_values)), dtype=bool)

for mean_index, target_mean in enumerate(target_mean_values):
    validity = np.array([
        [results[(target_mean, eps, k0)]["valid_domain_pattern"] for k0 in k0_values]
        for eps in eps_values
    ], dtype=bool)
    validity_cube[mean_index] = validity
    ax = axes.flat[mean_index]
    ax.imshow(validity, origin="lower", cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
    ax.set_xticks(range(len(k0_values)), [f"{v:.2f}" for v in k0_values], rotation=45)
    ax.set_yticks(range(len(eps_values)), [f"{v:.2f}" for v in eps_values])
    ax.set_xlabel("k0")
    ax.set_ylabel("eps")
    ax.set_title(f"target_mean={target_mean:+.2f}; valid={validity.sum()}/{validity.size}")
    for row in range(len(eps_values)):
        for col in range(len(k0_values)):
            ax.text(col, row, "✓" if validity[row, col] else "×", ha="center", va="center")
for ax in axes.flat[len(target_mean_values):]:
    ax.set_visible(False)
fig.suptitle("Coordinates producing at least one resolved magnetic domain", fontsize=14)

## 7. Valid volume in 3D

The green points are the accepted coordinate set. The grey points were simulated but rejected, including sparse isolated-bubble and uniform states. `valid_coordinates` can be sampled directly when generating a dataset.

In [ ]:
valid_coordinates = []
invalid_coordinates = []
for (target_mean, eps, k0), result in results.items():
    coordinate = (k0, eps, target_mean)
    (valid_coordinates if result["valid_domain_pattern"] else invalid_coordinates).append(coordinate)
valid_coordinates = np.asarray(valid_coordinates)
invalid_coordinates = np.asarray(invalid_coordinates)

fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection="3d")
if len(invalid_coordinates):
    ax.scatter(*invalid_coordinates.T, color="0.75", s=18, alpha=0.35, label="rejected")
if len(valid_coordinates):
    ax.scatter(*valid_coordinates.T, color="tab:green", s=42, alpha=0.9, label="valid domains")
ax.set_xlabel("k0")
ax.set_ylabel("eps")
ax.set_zlabel("target_mean")
ax.set_title("Valid domain-generating coordinate volume")
ax.legend()
print(f"Valid coordinates: {len(valid_coordinates)} / {len(results)}")

for target_mean in target_mean_values:
    accepted = [(k0, eps) for k0, eps, mean in valid_coordinates if np.isclose(mean, target_mean)]
    print(f"target_mean={target_mean:+.2f}: {len(accepted):2d} valid (k0, eps) pairs")

## 8. Sample continuous coordinates inside the valid volume

The sampled grid defines voxels in `(k0, eps, target_mean)` space. A conservative interior voxel is one whose eight corners are valid. The sampler chooses a voxel in proportion to its physical parameter-space volume, draws a continuous point inside it, generates that off-grid pattern, and verifies it with the same domain criterion. Failed interpolated points are rejected and redrawn. The returned coordinates therefore do not need to coincide with the original scan grid.

In [ ]:
def conservative_valid_cells(validity_cube):
    """Return bounds of voxels whose eight sampled corners are valid."""
    cells = []
    for mean_index in range(len(target_mean_values) - 1):
        for eps_index in range(len(eps_values) - 1):
            for k0_index in range(len(k0_values) - 1):
                corners = validity_cube[
                    mean_index:mean_index + 2,
                    eps_index:eps_index + 2,
                    k0_index:k0_index + 2]
                if np.all(corners):
                    cells.append(np.array([
                        [k0_values[k0_index], k0_values[k0_index + 1]],
                        [eps_values[eps_index], eps_values[eps_index + 1]],
                        [target_mean_values[mean_index], target_mean_values[mean_index + 1]],
                    ], dtype=float))
    return cells

valid_cells = conservative_valid_cells(validity_cube)
if not valid_cells:
    raise RuntimeError("No fully valid voxels were found; refine the scan grid or relax the validity thresholds.")
cell_volumes = np.array([np.prod(cell[:, 1] - cell[:, 0]) for cell in valid_cells])
cell_probabilities = cell_volumes / cell_volumes.sum()
print(f"Continuous sampling region contains {len(valid_cells)} conservative voxels")

def sample_valid_domain_coordinate(rng=None, max_attempts=100):
    """Draw and verify an off-grid coordinate inside the valid volume."""
    rng = np.random.default_rng() if rng is None else rng
    for attempt in range(1, max_attempts + 1):
        cell = valid_cells[rng.choice(len(valid_cells), p=cell_probabilities)]
        k0, eps, target_mean = rng.uniform(cell[:, 0], cell[:, 1])
        field, binary, metadata = generate_one(k0, eps, target_mean)
        passes_optional_filters, down_connectivity, up_connectivity = classify_domain_pattern(binary)
        morphology = classify_magnetic_domains(
            binary, min_area=bubble_min_area,
            max_eccentricity=bubble_max_eccentricity,
            min_circularity=bubble_min_circularity)
        has_resolved_domains = (
            morphology["bubble_count"] >= 1 or morphology["stripe_count"] >= 1)
        if passes_optional_filters and has_resolved_domains:
            return {
                "k0": float(k0), "eps": float(eps),
                "target_mean": float(target_mean),
                "field": field, "binary": binary,
                "down_fraction": down_fraction(binary),
                "down_connectivity": down_connectivity,
                "up_connectivity": up_connectivity,
                "morphology": morphology, "attempts": attempt,
            }
    raise RuntimeError(f"Could not verify a valid interpolated coordinate in {max_attempts} attempts")

In [ ]:
sampling_rng = np.random.default_rng(1234)
random_samples = [sample_valid_domain_coordinate(sampling_rng) for _ in range(8)]

fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection="3d")
ax.scatter(*valid_coordinates.T, color="tab:green", alpha=0.25, s=22, label="valid scan points")
sample_coordinates = np.array([[sample["k0"], sample["eps"], sample["target_mean"]]
                               for sample in random_samples])
ax.scatter(*sample_coordinates.T, color="tab:red", marker="*", s=130, label="verified random points")
ax.set_xlabel("k0"); ax.set_ylabel("eps"); ax.set_zlabel("target_mean")
ax.set_title("Off-grid samples inside the valid domain volume")
ax.legend()
for index, sample in enumerate(random_samples):
    print(index, {key: round(sample[key], 4) for key in ("k0", "eps", "target_mean")},
          "morphology=", sample["morphology"]["morphology"],
          "attempts=", sample["attempts"])

## 9. Stripe/bubble classification maps

The classifier binarizes at zero, labels both polarities, and analyzes the minority phase. A resolved component is a bubble when it is closed inside the image, has eccentricity below `bubble_max_eccentricity`, and circularity above `bubble_min_circularity`. Other resolved minority components are stripe-like. The first map shows the overall morphology; the second gives the actual bubble count.

In [ ]:
morphology_names = ["uniform", "noise", "bubbles", "stripes", "mixed"]
morphology_codes = {name: index for index, name in enumerate(morphology_names)}
morphology_colors = plt.matplotlib.colors.ListedColormap(
    ["black", "0.65", "tab:blue", "tab:orange", "tab:purple"])
ncols = 4
nrows = int(np.ceil(len(target_mean_values) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(12, 3 * nrows), squeeze=False)
for mean_index, target_mean in enumerate(target_mean_values):
    classification = np.array([
        [morphology_codes[results[(target_mean, eps, k0)]["morphology"]["morphology"]]
         for k0 in k0_values] for eps in eps_values])
    ax = axes.flat[mean_index]
    ax.imshow(classification, origin="lower", cmap=morphology_colors,
              vmin=-0.5, vmax=len(morphology_names)-0.5, aspect="auto")
    ax.set_xticks(range(len(k0_values)), [f"{v:.2f}" for v in k0_values], rotation=45)
    ax.set_yticks(range(len(eps_values)), [f"{v:.2f}" for v in eps_values])
    ax.set_xlabel("k0"); ax.set_ylabel("eps")
    ax.set_title(f"target_mean={target_mean:+.2f}")
for ax in axes.flat[len(target_mean_values):]: ax.set_visible(False)
handles = [plt.Line2D([0], [0], marker="s", linestyle="", color=morphology_colors(i),
                      label=name, markersize=9) for i, name in enumerate(morphology_names)]
fig.legend(handles=handles, loc="lower center", bbox_to_anchor=(0.5, -0.02), ncols=len(handles))
fig.suptitle("Minority-phase morphology classification", fontsize=14)

In [ ]:
fig, axes = plt.subplots(nrows, ncols, figsize=(12, 3 * nrows), squeeze=False)
max_bubbles = max(result["morphology"]["bubble_count"] for result in results.values())
for mean_index, target_mean in enumerate(target_mean_values):
    counts = np.array([
        [results[(target_mean, eps, k0)]["morphology"]["bubble_count"]
         for k0 in k0_values] for eps in eps_values])
    ax = axes.flat[mean_index]
    image = ax.imshow(counts, origin="lower", cmap="magma", vmin=0, vmax=max_bubbles, aspect="auto")
    ax.set_xticks(range(len(k0_values)), [f"{v:.2f}" for v in k0_values], rotation=45)
    ax.set_yticks(range(len(eps_values)), [f"{v:.2f}" for v in eps_values])
    ax.set_xlabel("k0"); ax.set_ylabel("eps")
    ax.set_title(f"target_mean={target_mean:+.2f}")
    for row in range(len(eps_values)):
        for col in range(len(k0_values)):
            ax.text(col, row, str(counts[row, col]), ha="center", va="center", fontsize=8)
for ax in axes.flat[len(target_mean_values):]: ax.set_visible(False)
fig.colorbar(image, ax=axes, label="bubble count", shrink=0.65)
fig.suptitle("Number of circle-like minority domains", fontsize=14)

## 10. Final analyzed parameter space

This is the combined representation. Saturated states are grey. Every domain-containing point uses a bivariate colour: increasing orange/red means more stripe-like components, while increasing blue means more bubbles. Mixed colours contain both. Counts are log-scaled in the colour calculation so a few very fragmented patterns do not compress the rest of the map. The adjacent 2D key gives the exact mapping.

In [ ]:
stripe_counts = np.array([result["morphology"]["stripe_count"] for result in results.values()])
bubble_counts = np.array([result["morphology"]["bubble_count"] for result in results.values()])
max_stripes = max(1, int(stripe_counts.max()))
max_bubbles = max(1, int(bubble_counts.max()))
stripe_endpoint = np.array([0.92, 0.25, 0.05])
bubble_endpoint = np.array([0.08, 0.35, 0.95])
empty_color = np.array([0.88, 0.88, 0.88])
saturated_color = np.array([0.45, 0.45, 0.45])

def stripe_bubble_color(stripe_count, bubble_count):
    """Map the two component counts to one bivariate RGB colour."""
    stripe_level = np.log1p(stripe_count) / np.log1p(max_stripes)
    bubble_level = np.log1p(bubble_count) / np.log1p(max_bubbles)
    total = stripe_level + bubble_level
    if total == 0:
        return empty_color
    mixture = (stripe_level * stripe_endpoint + bubble_level * bubble_endpoint) / total
    strength = max(stripe_level, bubble_level)
    return (1.0 - strength) * empty_color + strength * mixture

analyzed_points = []
point_colors = []
saturated_flags = []
for (target_mean, eps, k0), result in results.items():
    analysis = result["morphology"]
    minority_fraction = min(result["down_fraction"], 1.0 - result["down_fraction"])
    saturated = analysis["morphology"] == "uniform" or minority_fraction <= saturation_fraction_threshold
    analyzed_points.append((k0, eps, target_mean))
    saturated_flags.append(saturated)
    point_colors.append(saturated_color if saturated else stripe_bubble_color(
        analysis["stripe_count"], analysis["bubble_count"]))
analyzed_points = np.asarray(analyzed_points)
point_colors = np.asarray(point_colors)
saturated_flags = np.asarray(saturated_flags)

fig = plt.figure(figsize=(13, 7))
ax = fig.add_axes([0.04, 0.10, 0.64, 0.82], projection="3d")
ax.scatter(*analyzed_points.T, c=point_colors, s=48, depthshade=False, edgecolor="0.2", linewidth=0.25)
ax.set_xlabel("k0"); ax.set_ylabel("eps"); ax.set_zlabel("target_mean")
ax.set_title("Saturation and stripe/bubble content")

key_size = 150
stripe_axis = np.linspace(0, max_stripes, key_size)
bubble_axis = np.linspace(0, max_bubbles, key_size)
color_key = np.empty((key_size, key_size, 3))
for row, bubble_count in enumerate(bubble_axis):
    for col, stripe_count in enumerate(stripe_axis):
        color_key[row, col] = stripe_bubble_color(stripe_count, bubble_count)
key_ax = fig.add_axes([0.74, 0.23, 0.22, 0.50])
key_ax.imshow(color_key, origin="lower", aspect="auto",
              extent=[0, max_stripes, 0, max_bubbles])
key_ax.set_xlabel("stripe-like component count")
key_ax.set_ylabel("bubble count")
key_ax.set_title("Bivariate colour key")
key_ax.scatter([], [], color=saturated_color, marker="s", s=70, label="saturated")
key_ax.scatter([], [], color=empty_color, marker="s", s=70, edgecolor="0.4", label="no resolved components")
key_ax.legend(loc="upper left", bbox_to_anchor=(0, -0.16), frameon=False)
print(f"Saturated coordinates: {saturated_flags.sum()} / {len(saturated_flags)}")

## 11. Inspect one coordinate and its segmentation

In [ ]:
selected_k0 = k0_values[2]
selected_eps = eps_values[2]
selected_mean = target_mean_values[3]
selected = results[(selected_mean, selected_eps, selected_k0)]

analysis = selected["morphology"]
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(selected["field"], cmap="RdBu_r")
axes[0].set_title("continuous field")
axes[1].imshow(selected["binary"], cmap="gray", vmin=-1, vmax=1)
axes[1].set_title(f"raw binary; down={selected['down_fraction']:.1%}")
bubble_labels = {component["label"] for component in analysis["minority_components"]
                 if component["is_bubble"]}
bubble_mask = np.isin(analysis["minority_label_image"], list(bubble_labels))
axes[2].imshow(analysis["minority_label_image"], cmap="nipy_spectral")
if bubble_labels:
    axes[2].contour(bubble_mask, levels=[0.5], colors=["white"], linewidths=1.0)
axes[2].set_title(f"labels; bubbles={analysis['bubble_count']}, stripes={analysis['stripe_count']}")
for ax in axes: ax.set_axis_off()
print({"k0": selected_k0, "eps": selected_eps, "target_mean": selected_mean,
       "valid_domain_pattern": selected["valid_domain_pattern"],
       "morphology": analysis["morphology"],
       "bubble_count": analysis["bubble_count"],
       "stripe_count": analysis["stripe_count"]})

### Interpretation

This is a genuine three-parameter scan: `k0`, `eps`, and `target_mean` all act during evolution. By default, one resolved component is sufficient: a single bubble is a valid bubble state. Saturated and noise-only patterns remain invalid because they contain no resolved bubbles or stripes. Adjust the optional thresholds only if your physical definition of a domain pattern is stricter.